# PlantPal - Week 3 Data Pipeline
## Part A: Start with the Data

**Date:** 10 November 2026. **Scope:** A1-A4 only. Parts B and C have not been implemented.

This is a standard Python `.ipynb` notebook for VS Code/Jupyter, also compatible with Colab.
Saved outputs were produced by executing these cells locally, not in Colab. No model/API key is needed.
This notebook performs ingestion and inspection, not cleaning, chunking, embeddings, or model evaluation.

### Run in VS Code
Open the PlantPal repository folder, open this notebook, select a Python kernel, and choose **Run All**.
Run the setup cell again after a kernel restart. The dependency cell installs `pypdf==6.10.0` if needed.
Raw PDFs remain in ignored `data/raw/`; do not upload them or full extracted text to public GitHub.

### Run in Colab if required by the lecturer
Open this same notebook in Colab and select **Runtime > Run all**. It uses
`/content/week3_project/data/raw` and `data/processed` there. Files in `/content` are temporary.
The lab specifies Colab; a local run is not evidence of a connected Colab runtime.

The lab's optional dummy ZIP cell is deliberately omitted because actual PlantPal sources are used.


## A1 - Requirements and data needs

| Requirement ID | Needed information / feature | Candidate source | Why usable / constraint |
|---|---|---|---|
| FR1 | Documented light, watering principles, media and space guidance | S01 indoor guide; S02 houseplant brochure; relevant S03 sections | Official NParks material fits tropical gardening. Actual extraction and coverage must be inspected. Community plot planning and excluded chemical advice do not automatically become supported tasks. |
| FR2 | Document title, URL and one-based PDF page references | All three PDFs and their source register | Preserve page identity; a file hash identifies downloaded bytes, not accuracy or permission. |
| RAI1 | Verifiable source support and evidence of missing coverage | Original pages compared with extracted text | Layout loss, hidden text and icons can misrepresent evidence. A successful parser does not establish factual accuracy or support every plant-care question. |

These needs implement **A5 Approved Sources** and prepare inputs for **A4 Knowledge Access** in V1.
Later correctness and citation-support evaluation depend on reliable source/page associations.
O1 source inspection progresses here; user-needs research and a fully verified answer corpus remain pending.

**Selected data type:** unstructured PDF text. **Selected preparation pathway for Part B:** PDF/TXT (A).
Images are part of document layout, not a new image-recognition task. Other dataset candidates in
`docs/dataset_references.md` are not admitted by this notebook.


## A2 - Workspace and source access

The official [download directory](https://gardeningsg.nparks.gov.sg/learn-more-about-gardening/downloads/)
lists the three sources. [Terms of use](https://gardeningsg.nparks.gov.sg/terms-of-use/) were reviewed
on 10 November 2026. They describe personal non-commercial use and reserve other rights;
no open redistribution licence has been established. This notebook uses personal working copies
for inspection. Public outputs are limited to code, metadata, short samples and original observations.
This is not permission to redistribute PDFs, publish full extraction, train a model, or deploy a public service.

Use local copies when present. The acquisition cell fetches missing files from their explicit official URLs.
Alternatively download them yourself into `data/raw/` using the exact filenames in the register.


In [1]:
# Cell 1 - Workspace setup: RUN FIRST after every kernel restart.
from pathlib import Path
import sys, os, platform

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    BASE_DIR = Path('/content/week3_project')
else:
    # VS Code normally starts in the repository or notebooks directory.
    here = Path.cwd().resolve()
    candidates = [here, *here.parents]
    BASE_DIR = next((p for p in candidates if (p / 'docs/week2_requirements.md').is_file()), here)

RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR
assert RAW_DIR.is_dir() and PROCESSED_DIR.is_dir()
print('Environment:', 'Google Colab' if IN_COLAB else 'Local Python (VS Code/Jupyter compatible)')
print('Python:', platform.python_version())
print('Raw folder:', RAW_DIR)
print('Processed folder:', PROCESSED_DIR)


Environment: Local Python (VS Code/Jupyter compatible)
Python: 3.12.14
Raw folder: [local PlantPal workspace]\data\raw
Processed folder: [local PlantPal workspace]\data\processed


In [2]:
# Dependency setup: no model/API packages are needed for Part A.
import importlib.metadata
import subprocess
try:
    parser_version = importlib.metadata.version('pypdf')
except importlib.metadata.PackageNotFoundError:
    parser_version = None
if parser_version != '6.10.0':
    if 'pypdf' in sys.modules:
        raise RuntimeError('Restart the kernel before changing an already-imported pypdf version.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pypdf==6.10.0'])
from pypdf import PdfReader
print('Parser: pypdf', importlib.metadata.version('pypdf'))


Parser: pypdf 6.10.0


In [3]:
# Explicit source register. Expected hashes identify the bytes inspected on 10 November 2026.
SOURCES = [{'source_id': 'S01',
  'filename': 'introduction_to_indoor_gardening.pdf',
  'url': 'https://isomer-user-content.by.gov.sg/316/5be81ffb-fd47-444d-ab81-bf46b869dd7b/introduction_to_indoor_gardening.pdf',
  'title': 'Introduction to Indoor Gardening',
  'publisher': 'National Parks Board (NParks)',
  'expected_sha256': '21d6ae22ac7f4bbebfa38225d1fb942d2210376a119d350146fc15888a573533',
  'sample_pages': [5, 10],
  'access_date': '2026-11-10'},
 {'source_id': 'S02',
  'filename': 'know_10_houseplants.pdf',
  'url': 'https://isomer-user-content.by.gov.sg/316/43fdd8a1-d385-428f-8cc2-d5b9b36f6a38/know%2010%20house%20plants.pdf',
  'title': 'Know 10 Houseplants',
  'publisher': 'National Parks Board (NParks)',
  'expected_sha256': '2208509f38b4c3400270c06b3c20a7c2ddd1812ce4433a2fe3e4dd0b9bd60120',
  'sample_pages': [1, 2],
  'access_date': '2026-11-10'},
 {'source_id': 'S03',
  'filename': 'introduction_to_community_gardening.pdf',
  'url': 'https://isomer-user-content.by.gov.sg/316/e9f8a03b-d2aa-4f29-b16a-0678ea9ca5a9/introduction%20to%20community%20gardening.pdf',
  'title': 'Introduction to Community Gardening',
  'publisher': 'National Parks Board (NParks)',
  'expected_sha256': '33a4f05c6a55d110e2bcda45562c928b7edd440d750e214e1183717b30ab0096',
  'sample_pages': [5, 12],
  'access_date': '2026-11-10'}]
for source in SOURCES:
    print(source["source_id"], source["title"], "-> data/raw/" + source["filename"])


S01 Introduction to Indoor Gardening -> data/raw/introduction_to_indoor_gardening.pdf
S02 Know 10 Houseplants -> data/raw/know_10_houseplants.pdf
S03 Introduction to Community Gardening -> data/raw/introduction_to_community_gardening.pdf


In [4]:
# Acquire missing personal working copies. Never overwrite an existing source silently.
import hashlib, urllib.request
from datetime import datetime, timezone

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for source in SOURCES:
    path = RAW_DIR / source['filename']
    if not path.exists():
        request = urllib.request.Request(source['url'], headers={'User-Agent': 'PlantPal educational document inspection'})
        with urllib.request.urlopen(request, timeout=60) as response:
            content = response.read()
        if not content.startswith(b'%PDF'):
            raise ValueError(f"{source['source_id']}: download is not a PDF")
        path.write_bytes(content)
        print(source['source_id'], 'downloaded')
    else:
        print(source['source_id'], 'using existing personal copy')
    if not path.read_bytes().startswith(b'%PDF'):
        raise ValueError(f"{source['source_id']}: invalid PDF signature")
    actual = sha256_file(path)
    if actual != source['expected_sha256']:
        raise ValueError(f"{source['source_id']}: source bytes changed. Review the new version before updating the source register.")
print('Source signatures and fingerprints verified.')


S01 using existing personal copy
S02 using existing personal copy
S03 using existing personal copy
Source signatures and fingerprints verified.


In [5]:
# Cell 3 - Verify inputs recursively and select the intended files explicitly.
files = sorted(p for p in RAW_DIR.rglob('*') if p.is_file())
print('Input count:', len(files))
for path in files:
    print(path.relative_to(RAW_DIR).as_posix(), 'bytes:', path.stat().st_size)
assert files, 'No inputs: add the selected PDFs to data/raw.'
for source in SOURCES:
    assert (RAW_DIR / source['filename']).is_file(), source['filename']
print('All three selected PlantPal source files are accessible.')


Input count: 3
introduction_to_community_gardening.pdf bytes: 2450552
introduction_to_indoor_gardening.pdf bytes: 3178940
know_10_houseplants.pdf bytes: 1720974
All three selected PlantPal source files are accessible.


## A3 - Inspect actual content, page structure and ingestion limitations

The next cell extracts page text **in memory** and preserves one-based PDF page numbers.
It does not clean, chunk or export the corpus. Empty extraction is flagged for visual checking,
not automatically labelled a broken document or passed through OCR.
Two representative pages per source provide more than the lab's minimum of two source artefacts/subsets.
Short text samples are deliberately limited to nine words per inspected page.
Full extracted text stays in memory and is not written into this public notebook's outputs.


In [6]:
import json
READERS = {}
PAGE_TEXT = {}
INSPECTION = []
for source in SOURCES:
    path = RAW_DIR / source['filename']
    reader = PdfReader(path)
    if reader.is_encrypted:
        raise ValueError(f"{source['source_id']}: encrypted input requires an authorised readable copy")
    if not len(reader.pages):
        raise ValueError(f"{source['source_id']}: PDF has no pages")
    page_text = [page.extract_text() or '' for page in reader.pages]
    if not any(t.strip() for t in page_text):
        raise ValueError(f"{source['source_id']}: no extractable text; inspect visually before choosing a loader")
    READERS[source['source_id']] = reader
    PAGE_TEXT[source['source_id']] = page_text
    metadata = reader.metadata or {}
    record = {
        'source_id': source['source_id'], 'title': source['title'],
        'filename': source['filename'], 'url': source['url'], 'publisher': source['publisher'],
        'source_sha256': sha256_file(path), 'bytes': path.stat().st_size,
        'pdf_pages': len(reader.pages),
        'characters_per_page': [len(t) for t in page_text],
        'empty_text_pages': [i + 1 for i, t in enumerate(page_text) if not t.strip()],
        'pdf_creation_metadata': str(metadata.get('/CreationDate', 'not supplied')),
        'pdf_modification_metadata': str(metadata.get('/ModDate', 'not supplied')),
        'sample_pages': source['sample_pages'],
    }
    INSPECTION.append(record)
    print('\n', source['source_id'], source['title'])
    print('Pages:', record['pdf_pages'], '| characters:', sum(record['characters_per_page']))
    print('Empty-text pages:', record['empty_text_pages'])
    print('Creation metadata (not a verified publication date):', record['pdf_creation_metadata'])
    for page_no in source['sample_pages']:
        text = page_text[page_no - 1]
        excerpt = ' '.join(text.split()[:9])
        print(f'PDF page {page_no}: {len(text)} characters | sample: {excerpt} ...')
print('\nPASS: three readable PDFs inspected; original page identifiers retained.')



 S01 Introduction to Indoor Gardening
Pages: 24 | characters: 13273
Empty-text pages: [2]
Creation metadata (not a verified publication date): D:20201013090212+08'00'
PDF page 5: 638 characters | sample: Indoor Garden Considerations Space Identify a corner for you ...
PDF page 10: 1000 characters | sample: Plant Necessity - Media A good medium for your ...

 S02 Know 10 Houseplants
Pages: 2 | characters: 9439
Empty-text pages: []
Creation metadata (not a verified publication date): D:20230127115509Z
PDF page 1: 2810 characters | sample: For more information, visit our website at www.nparks.gov.sg/cib or ...
PDF page 2: 6629 characters | sample: Scientific Name Monstera obliqua Plant Characteristics The Window-leaf Monstera ...

 S03 Introduction to Community Gardening
Pages: 20 | characters: 7762
Empty-text pages: []
Creation metadata (not a verified publication date): D:20220602102149+08'00'
PDF page 5: 926 characters | sample: Soil Conditions Plants need good soil to grow well. ...


### Visual comparison recorded on 10 November 2026

- **S01:** PDF page 2 is a photograph with no visible text, explaining its zero extracted characters.
  Page 10 extracts an editorial image-placeholder sentence that is not visible in the rendered page.
  This is an actual extraction artefact to flag before using the text as evidence. Page 5 has multiple
  topic blocks; preserve their headings and boundaries in later preparation.
- **S02:** PDF page 2 displays ten plant profiles in a multi-column layout. Plain extraction starts with
  the bottom-right profile rather than the top-left profile, and several common-name headings appear
  together near the end. Text is readable, but sequence alone is unsafe for associating care text with a plant.
- **S03:** Page 5 mixes general growing guidance with community-garden planning. On page 12,
  care icons sit beside plant names but their meanings are not encoded as corresponding plain-text fields.
  Page 11 explains the icons. Page 9 includes advice outside PlantPal's excluded chemical guidance boundary.

Visual review is separate from automated extraction. These findings were obtained by viewing local
renderings of the originals; the copyrighted page images are not embedded in the public notebook.
On reruns, inspect these same pages in the original PDFs and verify whether the findings still apply.


In [7]:
# Reproducible structure checks for the inspected snapshot; no source text is modified.
checks = {
    'S01_page_2_has_no_extractable_text': not PAGE_TEXT['S01'][1].strip(),
    'S01_page_10_contains_editorial_placeholder': 'Can insert any relevant image here?' in PAGE_TEXT['S01'][9],
    'S02_page_2_starts_with_bottom_right_profile': PAGE_TEXT['S02'][1].lstrip().startswith('Scientific Name\nMonstera obliqua'),
    'S03_page_12_has_plant_name_text': 'Lettuce' in PAGE_TEXT['S03'][11],
}
for name, result in checks.items():
    print(name, ':', result)
assert all(checks.values()), 'Inspection snapshot changed: review the PDF and extraction before proceeding.'
print('These checks reproduce specific observations; they do not prove source accuracy or complete coverage.')


S01_page_2_has_no_extractable_text : True
S01_page_10_contains_editorial_placeholder : True
S02_page_2_starts_with_bottom_right_profile : True
S03_page_12_has_plant_name_text : True
These checks reproduce specific observations; they do not prove source accuracy or complete coverage.


## A4 - Data inventory checkpoint and preparation decision

See `docs/week3_data_inventory.md` for the source register, access review, file identities,
requirement mapping and observed limitations. The next cell saves a **metadata-only Part A report**,
not a prepared-data export or the Part B/C pipeline manifest.

**Part B decision:** use the PDF/TXT pathway. Preserve document and page provenance. Compare
layout-aware/heading-aware handling against simple extraction before choosing cleaning or chunking.
Flag hidden editorial text for exclusion with recorded justification. Do not turn care icons into
plant-specific claims without verifying their association. S03 remains a supplementary candidate;
ingestion does not admit all pages or excluded topics into PlantPal's answer scope.

No source content has been cleaned, chunked, embedded, or scored as a model answer in Part A.


In [8]:
REPORT_DIR = BASE_DIR / 'data' / 'inspection'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
REPORT = {
    'stage': 'week3-part-a-ingestion-and-inspection',
    'execution_environment': 'Google Colab' if IN_COLAB else 'local Python',
    'execution_timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'python_version': platform.python_version(),
    'pypdf_version': importlib.metadata.version('pypdf'),
    'raw_storage': 'data/raw/ (ignored by Git)',
    'records': INSPECTION,
    'snapshot_checks': checks,
    'note': 'Metadata only. No cleaned corpus, model evaluation or Part B/C pipeline results.',
}
report_path = REPORT_DIR / 'week3_part_a_inspection.json'
report_path.write_text(json.dumps(REPORT, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
assert json.loads(report_path.read_text(encoding='utf-8')) == REPORT
print('Saved and read back:', report_path.relative_to(BASE_DIR).as_posix())
print('Records:', len(REPORT['records']), '| raw PDF pages:', sum(r['pdf_pages'] for r in INSPECTION))


Saved and read back: data/inspection/week3_part_a_inspection.json
Records: 3 | raw PDF pages: 46


## Part A status

- A1: three existing Topic 2 requirements mapped to actual project PDFs.
- A2: local workspace initialised and all three inputs verified. Colab compatibility is provided;
  **a connected Colab execution has not been performed**.
- A3: three PDFs and six representative page subsets inspected, with separate visual comparisons.
- A4: inventory and metadata-only evidence prepared.
- Next: Part B preparation comparison, followed by Part C executable pipeline and workflow.

**Public sharing:** retain source notices on personal copies. Before saving/pushing a rerun, check
that no full text, source page images, private paths, credentials or personal input have been added
to outputs. The repository's raw/processed ignores do not remove content embedded in notebook outputs.
